In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
from IPython.display import clear_output

%pip install kagglehub catboost lightgbm tqdm -q

clear_output()
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm

%matplotlib inline

In [ ]:
# Task 1: Write your code here:
#Read the dataset Q1_data.csv using read_csv()
csv_path = os.path.join(path, "Q1_data.csv") # دمج المجلد واسم الملف

df = pd.read_csv(csv_path)

In [ ]:
# Task 2: Write your code here:
#inspect the first few rows using head()
df.head()

In [ ]:
# Task 3: Write your code here:
#Display dataset information using info()
df.info()

In [ ]:
# Task 4: Write your code here:
#Show statistical description using describe()
df.describe()

In [ ]:
# Task 5: Write your code here:
#Plot the target distribution (delivery_time)
#Your target is the column: "delivery_time".
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
#Drop the 'Order_ID' column from the data
df = df.drop(columns="Order_ID", axis=1)

In [ ]:
# Task 2: Write your code here:
#Handle missing values appropriately
#(Hint: I guess you want to have a closer look at
#the columns with missing values :) )

# 2. Do we have missing values?
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)

In [ ]:
# we have missing values , so drop
#df = df.drop(columns="Order_ID","Weather", "Traffic_Level","Time_of_Day",axis=1)
stat_cols= ["Weather", "Traffic_Level","Time_of_Day"]
# Drop rows with missing stat values
df_clean = df.dropna(subset=stat_cols).copy()
df = df.drop(columns="Weather", axis=1)
df = df.drop(columns="Traffic_Level", axis=1)
df = df.drop(columns="Time_of_Day", axis=1)
# Fill missing
#df_clean['Weather'] = df_clean['type2'].fillna('none')

In [ ]:
# Task 3: Write your code here:
# 4. Do we have duplicate samples?
def check_duplicates(df):
  duplicates = df.duplicated().sum() # حساب عدد الصفوف المكررة
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True) # inplace=True : يعني احذف في الجدول الاساسي، لا تنشأ واحد جديد
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 4: Write your code here:
#3. Do we have categorical columns?
categorical_cols = df.select_dtypes(include=["object"]).columns # اختيار الاعمدة ذات البيانات النصية

print("Categorical Columns:", list(categorical_cols))

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  df[col] = le.fit_transform(df[col])
  label_encoders[col] = le

df

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df.head()


In [ ]:
# Task 6: Write your code here:

In [ ]:
# Task 1: Write your code here:
X = df.drop("Delivery_Time", axis=1).astype(float) # نختار كل الاعمة الا التارقيت
y = df["Delivery_Time"].astype(float) # .astype(float) : يتاكد انه كل القيم اعداد عشرية

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

n_splits = 5 # K=5 Folds
# Stratified 5-Fold Cross-Validation, shuffled
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
    print(f"\nFold {fold_idx + 1}/{n_splits}")
'''
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]'''


In [ ]:
# Task 1: Write your code here:
from sklearn.ensemble import RandomForestClassifier

#**Define models with their hyperparameters**
models = {
  "Random Forest": RandomForestClassifier(
      n_estimators=320,  # Number of trees
      max_depth=4
  )
}

coeffs = {}
# استخراج المعاملات
coeffs['Lasso'] = models['LASSO Regression'].coef_
coeffs['Ridge'] = models['Ridge Regression'].coef_
# اعداد الرسم
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, coef) in enumerate(coeffs.items()):
  # Sort features by absolute coefficient value
  absolute_coef = np.abs(coef) # حساب اهمية كل ميزة بغض النظر عن الاشارة
  sorted_idx = np.argsort(absolute_coef) # ترتيب الميزات من الاقل اهمية الى الاكثر اهمية

  ax = axes[i]
  ax.barh(features[sorted_idx], coef[sorted_idx]) # رسم شريط افقي
  ax.set_title(f"{model_name} Coefficients")
  ax.set_xlabel("Coefficient Value (Impact)")

plt.tight_layout() # ترتيب
plt.show()

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: